# TOOLBOXLAP — Google Colab + Hugging Face / Ollama

One-cell experimental launcher for running an Ollama model inside a **Google Colab managed GPU runtime** and testing its OpenAI-compatible endpoint **locally inside the Colab runtime**.

> **Important:** Google currently disallows offering unrelated web services from managed Colab runtimes and also disallows connecting to remote proxies. Because of that, this companion notebook does **not** expose the runtime publicly with ngrok. Use the local API for interactive testing only.

**Website:** https://toolboxlap.com  |  **YouTube:** https://www.youtube.com/@TOOLBOXLAP-u1c  |  **GitHub:** https://github.com/toolboxlap-ve/TOOLBOXLAP-Colab-Ollama

In [ ]:
# TOOLBOXLAP — Google Colab Hugging Face / Ollama
# One-cell experimental launcher.
# IMPORTANT: Google Colab managed runtimes can restrict public service/proxy use.
# Use this notebook for testing only and follow Colab's current terms.

import os
import sys
import time
import json
import re
import shutil
import signal
import subprocess
import threading
from pathlib import Path
from urllib.parse import urlparse, unquote

DEFAULT_MODEL = "hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M"
PUBLIC_MODEL_ID = "toolboxlap"
OLLAMA_URL = "http://127.0.0.1:11434"
PROXY_PORT = 5000
HIGH_CONTEXT = 131072
FALLBACK_CONTEXT = 65536

print("=" * 72)
print("TOOLBOXLAP — Google Colab Hugging Face / Ollama")
print("=" * 72)

model_input = input(f"\nModel [ENTER = default: {DEFAULT_MODEL}]: " ).strip()
MODEL = model_input or DEFAULT_MODEL

def sh(cmd, check=True, env=None, capture=False):
    print("+", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check, text=True, env=env, capture_output=capture)

def ensure_pkg(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        sh([sys.executable, "-m", "pip", "install", "-q", pip_name])

def normalize_model(raw):
    value = raw.strip()
    if not value:
        return DEFAULT_MODEL
    if value.startswith("hf.co/"):
        return value
    parsed = urlparse(value)
    if parsed.netloc.lower() in {"huggingface.co", "www.huggingface.co"}:
        parts = [unquote(x) for x in parsed.path.split("/") if x]
        if len(parts) < 2:
            raise ValueError("Hugging Face URL must include owner and repository.")
        owner, repo = parts[0], parts[1]
        tag = ""
        if len(parts) >= 5 and parts[2] in {"blob", "resolve"}:
            filename = parts[-1]
            if filename.lower().endswith(".gguf"):
                tag = ":" + filename[:-5]
        q = re.search(r"(?:revision|tag)=([^&]+)", parsed.query)
        if q:
            tag = ":" + unquote(q.group(1))
        return f"hf.co/{owner}/{repo}{tag}"
    return value

MODEL = normalize_model(MODEL)
print(f"\nSelected backend model: {MODEL}", flush=True)

ensure_pkg("requests")
ensure_pkg("flask")

if shutil.which("zstd") is None:
    if shutil.which("apt-get") is None:
        raise RuntimeError("apt-get is unavailable; cannot install zstd.")
    sh(["apt-get", "update"])
    sh(["apt-get", "install", "-y", "zstd"])

if shutil.which("ollama") is None:
    sh(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])
if shutil.which("ollama") is None:
    raise RuntimeError("Ollama installation failed.")

import requests
def wait_http(url, timeout=90):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            r = requests.get(url, timeout=3)
            if r.ok:
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"Timed out waiting for {url}")

def start_ollama(context):
    env = os.environ.copy()
    env.update({"OLLAMA_FLASH_ATTENTION":"1","OLLAMA_KV_CACHE_TYPE":"q8_0","OLLAMA_NUM_PARALLEL":"1","OLLAMA_MAX_LOADED_MODELS":"1","OLLAMA_KEEP_ALIVE":"30m","OLLAMA_CONTEXT_LENGTH":str(context)})
    proc = subprocess.Popen(["ollama","serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    wait_http(OLLAMA_URL + "/api/tags")
    return proc

ollama_proc = None
context = HIGH_CONTEXT
try:
    ollama_proc = start_ollama(context)
    subprocess.run(["ollama","pull",MODEL], check=True)
except Exception:
    if ollama_proc:
        ollama_proc.terminate()
    context = FALLBACK_CONTEXT
    print("High context setup failed; retrying with fallback context 65536.")
    ollama_proc = start_ollama(context)
    subprocess.run(["ollama","pull",MODEL], check=True)

from flask import Flask, request, Response, jsonify
app = Flask(__name__)

@app.get("/health")
def health():
    return jsonify({"ok":True,"model":PUBLIC_MODEL_ID,"backend":MODEL,"context":context})

@app.get("/v1/models")
def models():
    return jsonify({"object":"list","data":[{"id":PUBLIC_MODEL_ID,"object":"model","owned_by":"toolboxlap"}]})

@app.post("/v1/chat/completions")
def chat():
    body = request.get_json(force=True, silent=True) or {}
    body["model"] = MODEL
    body.pop("reasoning_effort", None)
    if "max_tokens" not in body and "max_completion_tokens" not in body:
        body["max_tokens"] = 32768
    body.pop("max_completion_tokens", None)
    body.pop("tool_choice", None)
    body.pop("parallel_tool_calls", None)
    body.pop("store", None)
    body.pop("metadata", None)
    body.pop("service_tier", None)
    body.pop("logprobs", None)
    body.pop("top_logprobs", None)
    body.pop("modalities", None)
    body.pop("audio", None)
    r = requests.post(OLLAMA_URL + "/v1/chat/completions", json=body, stream=body.get("stream",False), timeout=600)
    if body.get("stream",False):
        return Response(r.iter_content(chunk_size=None), status=r.status_code, content_type=r.headers.get("content-type","text/event-stream"))
    return Response(r.content, status=r.status_code, content_type=r.headers.get("content-type","application/json"))

def run_flask():
    app.run(host="127.0.0.1", port=PROXY_PORT, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()
wait_http(f"http://127.0.0.1:{PROXY_PORT}/health")

print("\n✅ TOOLBOXLAP LOCAL API READY")
print(f"Local Base URL: http://127.0.0.1:{PROXY_PORT}/v1")
print(f"Model ID: {PUBLIC_MODEL_ID}")
print(f"Backend model: {MODEL}")
print(f"Context: {context}")

test = requests.get(f"http://127.0.0.1:{PROXY_PORT}/v1/models", timeout=15)
print(f"Local API test HTTP {test.status_code}")
print(test.text)
print("\n✅ EVERYTHING IS WORKING LOCALLY")